---

# Hydrological Response Modeling: A Time Series Analysis of Groundwater Levels in Columbus, Ohio

**Author:** Iaroslav Grushetskyi 

---

---
## Abstract

This study investigates groundwater-level dynamics and forecasting in Columbus, Ohio, using historical groundwater and climatic observations from 2010 to 2015. A time-series modeling framework is developed to characterize temporal dependence, seasonal variability, and the persistence of groundwater levels over time. Autoregressive Integrated Moving Average (ARIMA) and Seasonal ARIMA (SARIMA) models are evaluated to capture non-seasonal and seasonal temporal patterns, respectively, while Long Short-Term Memory (LSTM) neural networks are considered for modeling nonlinear and long-range temporal dependencies. Model performance is evaluated using out-of-sample forecasting metrics, including Mean Absolute Error (MAE), Root Mean Square Error (RMSE), and Mean Absolute Percentage Error (MAPE). The analysis focuses on identifying the temporal structure of groundwater-level fluctuations and determining which forecasting approach most effectively represents the observed dynamics. The results provide a time-series-based baseline for groundwater-level prediction in Columbus, Ohio, supporting improved understanding of aquifer behavior and informing localized groundwater-resource management under changing climatic conditions.

---

---
## Contents
1. [1. Introduction](#1-introduction)

2. [2. Methodology](#2-methodology)  
   - [2.1 Data Acquisition](#21-data-acquisition)  
   - [2.2 Climatological Patching](#22-climatological-patching)  
   - [2.3 Feature Engineering](#23-feature-engineering)  
   - [2.4 Visualizing Trends](#24-visualizing-trends)  

   - [2.5 Supervised Model: Random Forest Regressor](#25-supervised-model-random-forest-regressor)  
     - [2.5.1 Feature Definition](#251-feature-definition)  
     - [2.5.2 Train-Test Split](#252-train-test-split)  
     - [2.5.3 Model Training](#253-model-training)  
     - [2.5.4 Visual Comparison](#254-visual-comparison)  
     - [2.5.5 Model Comparison](#255-model-comparison)  
     - [2.5.6 Detrended Model](#256-detrended-model)  
     - [2.5.7 Summary](#257-summary)  

   - [2.6 Unsupervised Model: K-Means Clustering](#26-unsupervised-model-k-means-clustering)  
     - [2.6.1 Hydro State Features](#261-hydro-state-features)  
     - [2.6.2 Data Preparation](#262-data-preparation)  
     - [2.6.3 Scaling](#263-scaling)  
     - [2.6.4 Elbow Method](#264-elbow-method)  
     - [2.6.5 Model](#265-model)  
     - [2.6.6 Cluster Centroids](#266-cluster-centroids)  
     - [2.6.7 Logic-Based Mapping](#267-logic-based-mapping)  
     - [2.6.8 PCA](#268-pca)  
     - [2.6.9 PCA Loadings](#269-pca-loadings)  
     - [2.6.10 PCA Visualization](#2610-pca-visualization)  
     - [2.6.11 K-Means Summary](#2611-k-means-summary)  

3. [3. Results in the Context of RCPs](#3-results-in-the-context-of-rcps)  

4. [4. Limitations](#4-limitations)  

5. [5. Conclusion](#5-conclusion)  

6. [6. References](#6-references)

---

---
## 1. Introduction
The stability of groundwater resources is increasingly threatened by shifting climatic patterns. In the Midwestern United States, specifically the Scioto River Basin surrounding Columbus, Ohio, the interaction between surface precipitation and subsurface storage is a critical factor for agricultural and municipal planning. 

This project employs machine learning to bridge the gap between atmospheric observations and hydrological responses. By analyzing five years of high-frequency sensor data, we aim to identify the specific climate drivers—such as cumulative precipitation and thermal evapotranspiration—that dictate groundwater fluctuations. This research aligns with the broader framework of the **Representative Concentration Pathways (RCPs)**, as understanding historical climate sensitivity is the prerequisite for projecting how these resources will behave under future warming scenarios.

---

---
## 2. Methodology
### 2.1 Data Acquisition
Groundwater data was retrieved from the **USGS National Water Information System (NWIS)** for site `400540082540400` in Columbus, OH. Atmospheric data, including daily precipitation (PRCP) and temperature extremes (TMAX/TMIN), was merged with water level (WL) data to identify a continuous overlap period from **January 2010 to January 2015**.

---

In [1]:
# Importing the libraries
import pandas as pd, numpy as np, seaborn as sns
import dataretrieval.nwis as nwis
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

Geopandas not installed. Geometries will be flattened into pandas DataFrames.


In [2]:
# --- Global Sharpness & Academic Style --- 
plt.rcParams.update({ "figure.dpi": 250,
                      "pdf.fonttype": 42,
                      "font.family": "serif", 
                      "axes.labelsize": 11, 
                      "axes.titlesize": 12, 
                      "axes.titleweight": "bold", 
                      "axes.spines.top": False,
                      "axes.spines.right": False, 
                      "axes.grid": True,
                      "grid.alpha": 0.3, 
                      "grid.linestyle": "--", 
                      "legend.fontsize": 9, 
                      "legend.frameon": True, 
                      "legend.edgecolor": "0.8", 
                      "xtick.labelsize": 10, 
                      "ytick.labelsize": 10, 
                      "lines.linewidth": 1.5, 
                      "savefig.bbox": "tight",
                      "savefig.format": "pdf"})
# Set a professional color palette
sns.set_palette("colorblind")

In [3]:
def load_and_merge_climate_water(
    site='400540082540400',
    start='2010-01-01',
    end='2025-12-31',
    parameter='72019',
    atmospheric_path=None
):
    """
    Retrieve groundwater data from USGS, load atmospheric data,
    align overlapping dates, and return merged DataFrame.
    """

    # --- Retrieve water data ---
    water_df = nwis.get_record(
        sites=site,
        service='dv',
        start=start,
        end=end,
        parameterCd=parameter
    ).reset_index()

    # Format datetime
    water_df['datetime'] = pd.to_datetime(water_df['datetime']).dt.tz_localize(None).dt.normalize()

    # Rename + select columns
    water_df = water_df.rename(columns={
        'datetime': 'DATE',
        f'{parameter}_Maximum': 'WL'
    })[['DATE', 'WL']]

    # --- Load atmospheric data ---
    atmospheric_df = pd.read_csv(atmospheric_path)

    atmospheric_df['DATE'] = pd.to_datetime(atmospheric_df['DATE']).dt.tz_localize(None).dt.normalize()

    # --- Find overlap ---
    start_date = max(atmospheric_df['DATE'].min(), water_df['DATE'].min())
    end_date = min(atmospheric_df['DATE'].max(), water_df['DATE'].max())

    print(f"Overlap Period: {start_date} to {end_date}")

    # --- Filter datasets ---
    atmospheric_filtered = atmospheric_df[
        (atmospheric_df['DATE'] >= start_date) &
        (atmospheric_df['DATE'] <= end_date)
    ]

    water_filtered = water_df[
        (water_df['DATE'] >= start_date) &
        (water_df['DATE'] <= end_date)
    ]

    # --- Merge ---
    merged_df = pd.merge(atmospheric_filtered, water_filtered, on='DATE', how='inner')

    return merged_df

In [4]:
# Using load and merge function to put together the data
climate_water_df = load_and_merge_climate_water(
    atmospheric_path='/Users/Katia/Desktop/time_series_analysis_groundwater_levels/columbus_oh_atmospheric_data.csv'
)
print("The look of the first 5 rows of dataset:")
climate_water_df.head()

/var/folders/_6/cpkxcbtd4fg3c4_fkx5c2hk40000gn/T/ipykernel_18008/4020132743.py:14: DeprecationWarning: `nwis.get_record` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use the appropriate `waterdata.get_*()` for the service you need instead.
  water_df = nwis.get_record(


Overlap Period: 2010-01-01 00:00:00 to 2015-01-31 00:00:00
The look of the first 5 rows of dataset:


,STATION,DATE,PRCP,TMAX,TMIN,TOBS,WL
0,USC00331783,2010-01-01,0.00,36.0,20.0,20.0,21.25
1,USC00331783,2010-01-02,0.00,22.0,10.0,11.0,21.28
2,USC00331783,2010-01-03,0.00,20.0,5.0,6.0,21.26
3,USC00331783,2010-01-04,0.00,19.0,4.0,19.0,21.22
4,USC00331783,2010-01-05,0.05,24.0,17.0,18.0,21.25


In [5]:
print('The counts of missing values of the following features are:')
print('- precipitation:', climate_water_df['PRCP'].isna().sum())
print('- maximum temperature:', climate_water_df['TMAX'].isna().sum())
print('- minimum temperature:', climate_water_df['TMIN'].isna().sum())
print('- temperature at the time of observation:', climate_water_df['TOBS'].isna().sum())
print('- groundwater level:', climate_water_df['WL'].isna().sum())

The counts of missing values of the following features are:
- precipitation: 73
- maximum temperature: 104
- minimum temperature: 122
- temperature at the time of observation: 94
- groundwater level: 0


Since I have about 1,460 days of data (roughly 4 years), a gap of 10+ consecutive days is significant because it might skip an entire storm event or a seasonal shift. Let's check if the missing data is scattered (which would be easy to fix) or clustered in "large gaps" (harder to fix).

In [6]:
# Function to find the longest consecutive NaN (missing) stretch
def find_max_gap(df, column):
    # Creates a boolean series where True = NaN
    is_na = df[column].isna()
    
    # Groups consecutive True values together
    gaps = is_na.groupby((is_na != is_na.shift()).cumsum()).sum()
    
    return gaps.max()

# Check the main features
features = ['PRCP', 'TMAX', 'TMIN', 'WL']

print("Longest consecutive missing days:")
for col in features:
    max_gap = find_max_gap(climate_water_df, col)
    print(f"{col}: {max_gap} days")

Longest consecutive missing days:
PRCP: 5 days
TMAX: 30 days
TMIN: 30 days
WL: 0 days


I have a 30-day gap in temperature which is quite significant — it’s an entire month of the seasonal cycle. If that gap occurred in July, the model might "miss" the peak heat of the year, which drives groundwater evaporation (evapotranspiration).
Since precipitation gap is only 5 days, a simple Forward Fill or Zero Fill can manage that.

Now, let's pinpoint where excatly the longest stretches occur by identifying the "start" and "end" dates of these gaps. 

In [7]:
# Function to identify the longest stretch
def get_missing_ranges(df, column):
    # 1. Identify where data is missing
    is_na = df[column].isna()
    
    # 2. Group consecutive NaNs
    # This creates a unique ID for each block of missing values
    gap_id = (is_na != is_na.shift()).cumsum()
    
    # 3. Filter only for the gaps and group them
    gaps = df[is_na].groupby(gap_id[is_na])
    
    # 4. Extract start, end, and length
    summary = gaps['DATE'].agg(['min', 'max', 'count'])
    summary.columns = ['Start Date', 'End Date', 'Consecutive Days']
    
    return summary.reset_index(drop=True)

# Run it for your temperature columns
print("--- Temperature Max Gaps ---")
print(get_missing_ranges(climate_water_df, 'TMAX'))

print("\n--- Temperature Min Gaps ---")
print(get_missing_ranges(climate_water_df, 'TMIN'))

--- Temperature Max Gaps ---
   Start Date   End Date  Consecutive Days
0  2010-01-06 2010-01-06                 1
1  2010-01-21 2010-01-21                 1
2  2010-02-11 2010-02-11                 1
3  2010-02-18 2010-02-19                 2
4  2010-02-21 2010-02-28                 8
5  2010-03-02 2010-03-02                 1
6  2010-03-04 2010-03-04                 1
7  2010-03-09 2010-03-09                 1
8  2010-03-15 2010-03-16                 2
9  2010-03-24 2010-03-26                 3
10 2010-03-29 2010-03-31                 3
11 2010-04-08 2010-04-08                 1
12 2010-04-14 2010-04-14                 1
13 2010-04-22 2010-04-22                 1
14 2010-05-04 2010-05-04                 1
15 2010-05-10 2010-05-10                 1
16 2010-05-14 2010-05-14                 1
17 2010-05-20 2010-05-20                 1
18 2010-05-27 2010-05-27                 1
19 2010-06-16 2010-06-16                 1
20 2010-06-27 2010-06-27                 1
21 2010-06-29 2010-07-02 

The output above confirms a probable classic "station outage" pattern: we have a lot of tiny flickers in 2010 (1–2 days), but the elephant in the room is September 2013.
Both Max and Min temperatures are completely missing for the entire month of September 2013.

The possible problem with September 2013:\
In Columbus, September is a "transition month." It starts with late-summer heat and ends with the first touches of fall. If we use a simple linear interpolation across those 30 days, we’ll just get a perfectly straight, boring line that doesn't capture the actual temperature swings that drive groundwater changes.

The Solution - "Climatological Patching":\
Since we have data from 2010, 2011, and 2012, the best way to handle September 2013 is to fill it with the average of the other Septembers in our dataset. This preserves the September "shape".

---
### 2.2 Climatological Patching
Missing temperature observations during the September 2013 station outage imputed using calendar-day climatological means calculated exclusively from observations collected during 2010–2012.

---

In [12]:
# ============================================================
# TEMPERATURE IMPUTATION
# ============================================================

# Make sure DATE is datetime
climate_water_df['DATE'] = pd.to_datetime(climate_water_df['DATE'])

# Create a calendar-day key
climate_water_df['month_day'] = climate_water_df['DATE'].dt.strftime('%m-%d')


# ------------------------------------------------------------
# 1. Build climatology using ONLY 2010–2012
# ------------------------------------------------------------

historical_temperature = climate_water_df[
    climate_water_df['DATE'].dt.year.isin([2010, 2011, 2012])
].copy()

climatology = (
    historical_temperature
    .groupby('month_day')[['TMAX', 'TMIN']]
    .mean()
    .rename(columns={
        'TMAX': 'TMAX_CLIM',
        'TMIN': 'TMIN_CLIM'
    })
)


# ------------------------------------------------------------
# 2. Join climatological values to the full dataset
# ------------------------------------------------------------

climate_water_df = climate_water_df.join(
    climatology,
    on='month_day'
)


# ------------------------------------------------------------
# 3. Fill missing temperature observations using
#    historical calendar-day climatology
# ------------------------------------------------------------

climate_water_df['TMAX'] = climate_water_df['TMAX'].fillna(
    climate_water_df['TMAX_CLIM']
)

climate_water_df['TMIN'] = climate_water_df['TMIN'].fillna(
    climate_water_df['TMIN_CLIM']
)


# ------------------------------------------------------------
# 4. Fill any remaining short gaps using linear interpolation
# ------------------------------------------------------------

climate_water_df['TMAX'] = climate_water_df['TMAX'].interpolate(
    method='linear',
    limit_direction='both'
)

climate_water_df['TMIN'] = climate_water_df['TMIN'].interpolate(
    method='linear',
    limit_direction='both'
)


# ------------------------------------------------------------
# 5. Drop temporary columns
# ------------------------------------------------------------

climate_water_df = climate_water_df.drop(
    columns=['month_day', 'TMAX_CLIM', 'TMIN_CLIM', 'TOBS']
)

In [13]:
print("Now, let's examine the counts of missing values after performing filling methods to double check the results.")
print('The counts of missing values after filling methods, are:')
print('- precipitation:', climate_water_df['PRCP'].isna().sum())
print('- maximum temperature:', climate_water_df['TMAX'].isna().sum())
print('- minimum temperature:', climate_water_df['TMIN'].isna().sum())

Now, let's examine the counts of missing values after performing filling methods to double check the results.
The counts of missing values after filling methods, are:
- precipitation: 73
- maximum temperature: 0
- minimum temperature: 0


In [15]:
print("--- Precipitation Missing Ranges ---")
print(get_missing_ranges(climate_water_df, 'PRCP'))

--- Precipitation Missing Ranges ---
   Start Date   End Date  Consecutive Days
0  2010-01-21 2010-01-22                 2
1  2010-02-13 2010-02-13                 1
2  2010-03-02 2010-03-02                 1
3  2010-03-04 2010-03-04                 1
4  2010-03-09 2010-03-10                 2
5  2010-03-15 2010-03-17                 3
6  2010-04-08 2010-04-09                 2
7  2010-04-14 2010-04-14                 1
8  2010-04-22 2010-04-22                 1
9  2010-05-20 2010-05-21                 2
10 2010-05-27 2010-05-28                 2
11 2010-07-01 2010-07-02                 2
12 2010-07-18 2010-07-22                 5
13 2010-07-26 2010-07-27                 2
14 2010-08-12 2010-08-13                 2
15 2010-08-16 2010-08-16                 1
16 2010-10-09 2010-10-09                 1
17 2010-10-29 2010-10-30                 2
18 2010-11-04 2010-11-05                 2
19 2010-11-15 2010-11-16                 2
20 2010-11-26 2010-11-27                 2
21 2010-12-16 201

In [16]:
print("--- Individual Missing Precipitation Dates ---")

missing_prcp = climate_water_df[
    climate_water_df['PRCP'].isna()
][['DATE', 'PRCP']]

print(missing_prcp.to_string(index=False))

--- Individual Missing Precipitation Dates ---
      DATE  PRCP
2010-01-21   NaN
2010-01-22   NaN
2010-02-13   NaN
2010-03-02   NaN
2010-03-04   NaN
2010-03-09   NaN
2010-03-10   NaN
2010-03-15   NaN
2010-03-16   NaN
2010-03-17   NaN
2010-04-08   NaN
2010-04-09   NaN
2010-04-14   NaN
2010-04-22   NaN
2010-05-20   NaN
2010-05-21   NaN
2010-05-27   NaN
2010-05-28   NaN
2010-07-01   NaN
2010-07-02   NaN
2010-07-18   NaN
2010-07-19   NaN
2010-07-20   NaN
2010-07-21   NaN
2010-07-22   NaN
2010-07-26   NaN
2010-07-27   NaN
2010-08-12   NaN
2010-08-13   NaN
2010-08-16   NaN
2010-10-09   NaN
2010-10-29   NaN
2010-10-30   NaN
2010-11-04   NaN
2010-11-05   NaN
2010-11-15   NaN
2010-11-16   NaN
2010-11-26   NaN
2010-11-27   NaN
2010-12-16   NaN
2010-12-17   NaN
2011-02-02   NaN
2011-03-12   NaN
2011-03-26   NaN
2011-05-06   NaN
2011-06-08   NaN
2011-06-11   NaN
2011-09-08   NaN
2011-09-19   NaN
2011-09-20   NaN
2011-09-23   NaN
2011-10-12   NaN
2011-10-19   NaN
2011-10-27   NaN
2011-12-22   NaN
2

In [17]:
# ============================================================
# PRECIPITATION IMPUTATION
# ============================================================

# Make sure data are sorted chronologically
climate_water_df = climate_water_df.sort_values('DATE').copy()

# Keep track of which precipitation values were originally missing
climate_water_df['PRCP_IMPUTED'] = climate_water_df['PRCP'].isna()

print(
    "Precipitation values to be imputed:",
    climate_water_df['PRCP_IMPUTED'].sum()
)

# ------------------------------------------------------------
# Interpolate short precipitation gaps using time
# ------------------------------------------------------------

climate_water_df = climate_water_df.set_index('DATE')

climate_water_df['PRCP'] = climate_water_df['PRCP'].interpolate(
    method='time',
    limit=5,
    limit_direction='both'
)

# ------------------------------------------------------------
# Check remaining missing values
# ------------------------------------------------------------

print(
    "Remaining missing precipitation values:",
    climate_water_df['PRCP'].isna().sum()
)

# Restore DATE as a column
climate_water_df = climate_water_df.reset_index()

Precipitation values to be imputed: 73
Remaining missing precipitation values: 0


Indeed we see no presence of missing values after "patching" work. 
One Last Logic Check - The "September 2013" Verification:
Since I manually patched that 30-day gap, let's take a quick look at the data to make sure it looks like a real September and not just a flat line.

In [18]:
model_df = climate_water_df[
    ['DATE', 'WL', 'PRCP', 'TMAX', 'TMIN']
].copy()

model_df = model_df.sort_values('DATE').reset_index(drop=True)

print("Final missing-value check:")
print(model_df.isna().sum())

Final missing-value check:
DATE    0
WL      0
PRCP    0
TMAX    0
TMIN    0
dtype: int64


In [19]:
print("Start:", model_df['DATE'].min())
print("End:", model_df['DATE'].max())
print("Number of observations:", len(model_df))

print("\nDuplicate dates:")
print(model_df['DATE'].duplicated().sum())

print("\nDate frequency:")
print(model_df['DATE'].diff().value_counts().head())

Start: 2010-01-01 00:00:00
End: 2015-01-31 00:00:00
Number of observations: 1463

Duplicate dates:
0

Date frequency:
DATE
1 days    1314
2 days      85
3 days      38
4 days       9
5 days       7
Name: count, dtype: int64


In [22]:
print("Original WL observations:")
print("Number of observations:", len(climate_water_df))

print("\nWL missing values:")
print(climate_water_df['WL'].isna().sum())

print("\nDate range:")
print(climate_water_df['DATE'].min())
print(climate_water_df['DATE'].max())

Original WL observations:
Number of observations: 1463

WL missing values:
0

Date range:
2010-01-01 00:00:00
2015-01-31 00:00:00


In [23]:
wl_dates = (
    climate_water_df[['DATE', 'WL']]
    .sort_values('DATE')
    .copy()
)

wl_dates['DAY_GAP'] = wl_dates['DATE'].diff().dt.days

print("\nLargest gaps in groundwater observations:")
print(
    wl_dates.nlargest(20, 'DAY_GAP')
    [['DATE', 'DAY_GAP']]
)


Largest gaps in groundwater observations:
           DATE  DAY_GAP
914  2013-06-01     83.0
944  2013-08-01     32.0
1340 2014-10-01     31.0
869  2013-01-05      9.0
618  2011-12-22      8.0
398  2011-03-12      6.0
432  2011-04-30      6.0
809  2012-10-01      6.0
850  2012-11-27      6.0
478  2011-07-01      5.0
711  2012-04-28      5.0
713  2012-05-04      5.0
756  2012-07-04      5.0
771  2012-07-24      5.0
780  2012-08-11      5.0
892  2013-02-09      5.0
368  2011-01-22      4.0
383  2011-02-12      4.0
428  2011-04-21      4.0
521  2011-08-24      4.0
